# 1. Compute Security — Bastion, JIT, AKS, Containers, Disk Encryption

## Azure Bastion — implementation

```bash
# Create AzureBastionSubnet (must be /26 or larger, must be named exactly this)
az network vnet subnet create -g rg-prod --vnet-name vnet-prod \
  -n AzureBastionSubnet --address-prefixes 10.0.3.0/26

# Create public IP for Bastion
az network public-ip create -g rg-prod -n bastion-pip \
  --sku Standard --allocation-method Static

# Create Bastion host
az network bastion create -g rg-prod -n bastion-prod \
  --vnet-name vnet-prod --public-ip-address bastion-pip \
  --sku Standard  # Standard enables file transfer, IP-based connections
```

### Bastion SKUs

| Feature | Basic | Standard |
|---------|-------|----------|
| RDP/SSH to VMs in same VNet | ✅ | ✅ |
| RDP/SSH to VMs in peered VNets | ❌ | ✅ |
| Connect by IP address | ❌ | ✅ |
| File upload/download | ❌ | ✅ |
| Shareable link | ❌ | ✅ |

## JIT (Just-In-Time) VM Access

Requires **Microsoft Defender for Servers**. Management ports are blocked by NSG rules. When an admin requests access, Defender for Cloud temporarily opens the port for a specific IP and duration.

```bash
# Enable JIT on a VM
az rest --method PUT \
  --uri 'https://management.azure.com/subscriptions/{sub}/resourceGroups/rg-prod/providers/Microsoft.Security/jitNetworkAccessPolicies/default?api-version=2020-01-01' \
  --body '{
    "kind": "Basic",
    "properties": {
      "virtualMachines": [{
        "id": "/subscriptions/.../virtualMachines/my-vm",
        "ports": [
          {"number": 22, "protocol": "TCP", "allowedSourceAddressPrefix": "*", "maxRequestAccessDuration": "PT3H"},
          {"number": 3389, "protocol": "TCP", "allowedSourceAddressPrefix": "*", "maxRequestAccessDuration": "PT3H"}
        ]
      }]
    }
  }'

# Request access (opens port for your IP for 1 hour)
az security jit-policy request --resource-group rg-prod --name default \
  --vm my-vm --port 22 --duration PT1H
```

### Bastion vs JIT — when to use which

| | Bastion | JIT |
|-|---------|-----|
| Removes public IP from VM | ✅ | No (VM still has public IP, just ports are closed) |
| Requires agent | No | No |
| Protocol | RDP/SSH via browser | Native RDP/SSH client |
| Best for | All scenarios (preferred) | Legacy apps that need native RDP |

In [ ]:
import json
from datetime import datetime, timedelta

# Simulate JIT access workflow
JIT_POLICIES = {
    'my-vm': {
        'ports': [{'number': 22, 'max_duration': 'PT3H'}, {'number': 3389, 'max_duration': 'PT3H'}],
        'active_requests': [],
    },
}

def request_jit(vm: str, port: int, requester_ip: str, duration_hours: int):
    policy = JIT_POLICIES.get(vm)
    if not policy:
        return {'status': '❌ VM not enrolled in JIT'}
    port_config = next((p for p in policy['ports'] if p['number'] == port), None)
    if not port_config:
        return {'status': f'❌ Port {port} not configured in JIT policy'}
    
    now = datetime.now()
    return {
        'status': '✅ Access granted',
        'vm': vm,
        'port': port,
        'source_ip': requester_ip,
        'nsg_rule_added': f'JIT-Allow-{port}-{requester_ip.replace(".","-")}',
        'active_until': (now + timedelta(hours=duration_hours)).strftime('%H:%M'),
        'auto_cleanup': 'NSG rule will be removed automatically after expiration',
    }

print('=== JIT VM Access Request ===\n')
print('Before: Port 22 is BLOCKED by NSG (deny rule priority 100)\n')

result = request_jit('my-vm', 22, '203.0.113.50', 1)
print(json.dumps(result, indent=2))

print('\nAfter: NSG temporarily has a high-priority ALLOW rule for 203.0.113.50:22')
print('When the hour expires, the rule is automatically removed.')

## AKS Security

### Key AKS security configurations

| Feature | CLI | Purpose |
|---------|-----|---------|
| **Network policy** | `--network-policy calico` | Pod-to-pod firewalling |
| **Private cluster** | `--enable-private-cluster` | API server on private IP only |
| **Entra integration** | `--enable-aad` | Authenticate with Entra ID |
| **Workload identity** | `--enable-oidc-issuer --enable-workload-identity` | Federated MI for pods |
| **Azure Policy** | `--enable-addons azure-policy` | Enforce OPA Gatekeeper policies |
| **Defender for Containers** | Enable via Defender for Cloud | Runtime protection, image scanning |
| **ACR integration** | `--attach-acr <acr-name>` | Pull images without secrets |

```bash
# Create secure AKS cluster
az aks create -g rg-prod -n aks-prod \
  --enable-private-cluster \
  --enable-aad --aad-admin-group-object-ids <group-id> \
  --enable-oidc-issuer --enable-workload-identity \
  --network-policy calico \
  --enable-addons azure-policy \
  --attach-acr myacr
```

## Disk encryption options

| Method | What it encrypts | Key management |
|--------|-----------------|----------------|
| **SSE (default)** | Data at rest on managed disks | Platform-managed key (PMK) |
| **SSE with CMK** | Same + customer controls the key | Key Vault / Managed HSM |
| **ADE** | OS + data disks (BitLocker/DM-Crypt) | Key Vault |
| **Encryption at host** | Temp disk + OS/data disk caches | PMK or CMK |
| **Confidential disk** | VM memory + disk (TEE) | Platform / customer |

**Exam tip**: SSE is always on. ADE adds OS-level encryption (BitLocker). "Encryption at host" covers temp disks and caches that SSE misses.

```bash
# Enable ADE on a VM
az vm encryption enable -g rg-prod -n my-vm \
  --disk-encryption-keyvault /subscriptions/.../vaults/my-kv \
  --volume-type All
```

---
## Summary

| Feature | Key implementation detail |
|---------|-------------------------|
| **Bastion** | AzureBastionSubnet (/26+), Standard SKU for peered VNets |
| **JIT** | Requires Defender for Servers. Temporary NSG rules. |
| **AKS** | Private cluster + Entra auth + workload identity + network policy |
| **ACR** | Attach to AKS for secretless pulls. Enable content trust for signing. |
| **Disk encryption** | SSE (default) + ADE (OS-level) + encryption at host (caches) |

**Next**: [Notebook 2 — Storage Security](02_storage_security.ipynb)